In [ ]:
# --- Step 0: Environment Dependencies Installation ---
# Installs standard PyTorch core components if not already present in the local kernel environment
import sys
!{sys.executable} -m pip install --quiet torch
import torch
import torch.nn as nn
import torch.nn.functional as F

# --- Data Ingestion Setup ---
# B = Batch Size (64), N = Input Feature Count (10)
B = 64
N = 10
C = 3  # Target Classes (e.g., Customer Churn Risk Level: Low, Medium, High)

print(f"[INFO] Initializing synthetic tabular data loader infrastructure...")
torch.manual_seed(42)  # Ensuring deterministic initialization across environments

# Generate synthetic tabular classification features [B, N]
X = torch.randn(B, N)

# Generate synthetic target class indices [B] where each value is in range [0, C-1]
y_true_indices = torch.randint(0, C, (B,), dtype=torch.long)

# FIX: Changed hyphen to underscore so PyTorch recognizes the functional call
y_true_onehot = F.one_hot(y_true_indices, num_classes=C).float()

print(f"-> Success: Input Feature Batch (X) Shape: {list(X.shape)}")
print(f"-> Success: Ground Truth Target Class Indices (y_true_indices) Shape: {list(y_true_indices.shape)}")
print(f"-> Success: One-Hot Encoded Targets (y_true_onehot) Shape: {list(y_true_onehot.shape)}")

In [ ]:
class CustomFeedforwardNet(nn.Module):
    def __init__(self, input_dim, hidden1_dim, hidden2_dim, output_dim):
        super().__init__()
        # M1, M2 represent hidden dimensions. 
        # We explicitly set requires_grad=True to build the Autograd tracking tape
        
        # Layer 1 Parameters
        self.W1 = nn.Parameter(torch.randn(input_dim, hidden1_dim) * 0.1, requires_grad=True)
        self.b1 = nn.Parameter(torch.zeros(hidden1_dim), requires_grad=True)
        
        # Layer 2 Parameters
        self.W2 = nn.Parameter(torch.randn(hidden1_dim, hidden2_dim) * 0.1, requires_grad=True)
        self.b2 = nn.Parameter(torch.zeros(hidden2_dim), requires_grad=True)
        
        # Output Layer Parameters (No activation function applied to raw outputs)
        self.W3 = nn.Parameter(torch.randn(hidden2_dim, output_dim) * 0.1, requires_grad=True)
        self.b3 = nn.Parameter(torch.zeros(output_dim), requires_grad=True)

    def forward_manual(self, x):
        """Executes forward propagation returning intermediate shapes for validation assertions."""
        # Layer 1: Linear Transformation + ReLU Activation
        self.Z1 = x @ self.W1 + self.b1
        self.A1 = torch.relu(self.Z1)
        
        # Layer 2: Linear Transformation + ReLU Activation
        self.Z2 = self.A1 @ self.W2 + self.b2
        self.A2 = torch.relu(self.Z2)
        
        # Output Layer: Raw Unnormalized Logits (OMIT ReLU to prevent numeric stability failure)
        self.logits = self.A2 @ self.W3 + self.b3
        return self.logits

In [ ]:
# Instantiate model: 10 input features -> 32 nodes -> 16 nodes -> 3 output classification choices
model = CustomFeedforwardNet(input_dim=10, hidden1_dim=32, hidden2_dim=16, output_dim=3)

# Execute single baseline pass
logits = model.forward_manual(X)

print("[CHECKPOINT] Running dimensional alignment evaluation tests...")
# Guardrail Assertions based on matrix dim properties: [B, N] x [N, M] -> [B, M]
assert list(model.A1.shape) == [64, 32], f"Layer 1 dimension mismatch! Found: {list(model.A1.shape)}"
assert list(model.A2.shape) == [64, 16], f"Layer 2 dimension mismatch! Found: {list(model.A2.shape)}"
assert list(logits.shape) == [64, 3], f"Output Layer logit dimension mismatch! Found: {list(logits.shape)}"
print("-> All shape-assertion checkpoints passed cleanly! Hidden layers are mathematically aligned.")

In [ ]:
# --- Step 3: Manual Cross Entropy Calculation ---
# Softmax normalization formula: exp(x) / sum(exp(x)) computed along the class dim (-1)
exp_logits = torch.exp(logits)
softmax_probs = exp_logits / torch.sum(exp_logits, dim=-1, keepdim=True)

# Compute loss formula manually: L = -1/B * sum(y_true * log(y_pred))
manual_loss = -torch.mean(torch.sum(y_true_onehot * torch.log(softmax_probs), dim=-1))
print(f"\nStep 3: Calculated Manual Scalar Loss Evaluation Value: {manual_loss.item():.4f}")

# --- Step 4: Gradient Comparison Validation Block ---
# Derivative of cross entropy loss with respect to output logits is: dL/dLogits = (probabilities - labels) / BatchSize
d_logits = (softmax_probs - y_true_onehot) / B

# Using chain rule, derivative with respect to output weight W3 is: A2^T @ d_logits
manual_grad_W3 = model.A2.t() @ d_logits

# Native Autograd Evaluation 
manual_loss.backward()
autograd_grad_W3 = model.W3.grad

# Precision Verification Testing
gradients_match = torch.allclose(manual_grad_W3, autograd_grad_W3, atol=1e-5)
print(f"Step 4: Do manual gradients match native autograd metrics? {gradients_match}")
print(f"-> Manual Weight 3 Grad Sample (First 2x2 Element Array):\n{manual_grad_W3[:2, :2]}")
print(f"-> Autograd W3.grad Sample (First 2x2 Element Array):\n{autograd_grad_W3[:2, :2]}")

In [ ]:
# Reset network states completely for a clean 100-epoch execution runtime test
torch.manual_seed(42)
model = CustomFeedforwardNet(input_dim=10, hidden1_dim=32, hidden2_dim=16, output_dim=3)
loss_fn = nn.CrossEntropyLoss()  # Automatically combines LogSoftmax + NLLLoss safely under the hood

learning_rate = 0.5
epochs = 100
print(f"\nStep 5: Beginning baseline optimization sequence loop...")

for epoch in range(1, epochs + 1):
    # Forward Pass through raw logits
    logits = model.forward_manual(X)
    loss = loss_fn(logits, y_true_indices)
    
    # Critical Step: Zero out the gradient tracking buffers from the previous iteration
    if model.W3.grad is not None:
        model.W1.grad.zero_()
        model.b1.grad.zero_()
        model.W2.grad.zero_()
        model.b2.grad.zero_()
        model.W3.grad.zero_()
        model.b3.grad.zero_()
        
    # Execute the backward propagation run
    loss.backward()
    
    # Manual Gradient Descent Update Rule wrapped safely inside tracking protection context
    with torch.no_grad():
        model.W1 -= learning_rate * model.W1.grad
        model.b1 -= learning_rate * model.b1.grad
        model.W2 -= learning_rate * model.W2.grad
        model.b2 -= learning_rate * model.b2.grad
        model.W3 -= learning_rate * model.W3.grad
        model.b3 -= learning_rate * model.b3.grad

    if epoch % 10 == 0 or epoch == 1:
        print(f"-> Epoch [{epoch:3d}/{epochs}] | Optimization Convergence Tracking Target Loss: {loss.item():.5f}")

print("\n[SUCCESS] Baseline training loop executed completely without runtime compilation errors.")